# Example Conversation Viewer

This notebook displays a formatted conversation from a molecule optimization experiment JSON file.

In [1]:
import json
from pathlib import Path
from IPython.display import display, HTML

## Load Conversation

Specify the path to the conversation JSON file below.

In [2]:
# Path to the conversation JSON file
json_file_path = "../data/results/similarity_qed/relevant_molecule/full_xai/sim_qed_relevant_full_rep1_conversation_20251208_114854.json"

# Load the data
with open(json_file_path, 'r') as f:
    data = json.load(f)

print(f"Loaded: {Path(json_file_path).name}")
print(f"Experiment: {data.get('experiment', 'N/A')}")
print(f"Timestamp: {data.get('timestamp', 'N/A')}")
print(f"Iterations: {data.get('iterations', 'N/A')}")
print(f"Messages: {len(data.get('conversation', []))}")

Loaded: sim_qed_relevant_full_rep1_conversation_20251208_114854.json
Experiment: sim_qed_relevant_full_rep1
Timestamp: 20251208_114854
Iterations: 50
Messages: 101


## Formatted Conversation Display

In [3]:
def format_conversation_html(data: dict) -> str:
    """Format the conversation data as HTML for display."""
    html_parts = []
    
    # CSS styles
    html_parts.append('''
    <div style="font-family: system-ui, sans-serif; color: #000000; background-color: #ffffff; padding: 20px; border-radius: 8px;">
    <style>
        h1, h2, h3, h4, h5, h6 { color: #000000 !important; font-weight: bold; }
        h1 { font-size: 32px; margin-bottom: 15px; }
        h3 { font-size: 20px; }
        p { color: #000000; }
        strong { color: #000000; font-weight: bold; }
        .conv-message { margin-bottom: 20px; }
        .conv-message h3 { color: #000000 !important; margin-bottom: 8px; }
        .conv-message pre { 
            background-color: #f8f9fa; 
            border: 1px solid #ddd;
            padding: 15px; 
            border-radius: 5px; 
            overflow-x: auto; 
            color: #000000;
            font-size: 13px;
            line-height: 1.5;
            white-space: pre-wrap;
            word-wrap: break-word;
        }
        .system-msg pre { border-left: 4px solid #95a5a6; }
        .human-msg pre { border-left: 4px solid #3498db; }
        .ai-msg pre { border-left: 4px solid #2ecc71; }
        .highlight-box {
            margin: 10px 0; 
            padding: 15px; 
            background-color: #fff9e6; 
            border: 2px solid #ffa500; 
            border-radius: 5px;
        }
        .highlight-box p { margin: 5px 0; color: #000000; }
        .highlight-box strong { color: #d97706; font-weight: bold; }
        .smiles-code {
            display: block; 
            background-color: #ffffff; 
            padding: 10px; 
            border-radius: 4px; 
            overflow-x: auto; 
            word-break: break-all;
            font-family: 'Courier New', monospace;
            color: #000000;
            font-weight: bold;
            border: 1px solid #dee2e6;
        }
        .config-box {
            background-color: #f0f0f0;
            padding: 15px;
            border-radius: 5px;
            margin-bottom: 20px;
        }
    </style>
    ''')
    
    # Header
    experiment_name = data.get('experiment', 'Unknown Experiment')
    html_parts.append(f'<h1>🧬 {experiment_name}</h1>')
    html_parts.append(f'<p><strong>Timestamp:</strong> {data.get("timestamp", "N/A")}</p>')
    html_parts.append(f'<p><strong>Iterations:</strong> {data.get("iterations", "N/A")}</p>')
    html_parts.append(f'<p><strong>Total Messages:</strong> {len(data.get("conversation", []))}</p>')
    
    # Config if available
    if 'config' in data:
        html_parts.append('<details><summary><strong>📋 Experiment Configuration</strong></summary>')
        html_parts.append('<div class="config-box">')
        html_parts.append(f'<pre>{json.dumps(data["config"], indent=2)}</pre>')
        html_parts.append('</div></details>')
    
    html_parts.append('<hr style="border: 1px solid #ccc; margin: 20px 0;">')
    
    # Conversation messages
    for i, msg in enumerate(data.get('conversation', [])):
        role = msg.get('role', 'Unknown')
        content = msg.get('content', '')
        
        if role == "SystemMessage":
            html_parts.append('<div class="conv-message system-msg">')
            html_parts.append(f'<h3>📋 Message {i+1}: System Prompt</h3>')
            html_parts.append(f'<pre>{content}</pre>')
            html_parts.append('</div>')
        
        elif role == "HumanMessage":
            html_parts.append('<div class="conv-message human-msg">')
            html_parts.append(f'<h3>👤 Message {i+1}: Human (Task/Feedback)</h3>')
            html_parts.append(f'<pre>{content}</pre>')
            html_parts.append('</div>')
        
        elif role == "AIMessage":
            html_parts.append('<div class="conv-message ai-msg">')
            html_parts.append(f'<h3>🤖 Message {i+1}: AI Response</h3>')
            
            # Try to parse as JSON for better formatting
            try:
                parsed = json.loads(content)
                html_parts.append(f'<pre>{json.dumps(parsed, indent=2)}</pre>')
                
                # Show key info with proper formatting
                if 'smiles' in parsed and 'reason' in parsed:
                    html_parts.append('<div class="highlight-box">')
                    html_parts.append('<p><strong>SMILES:</strong></p>')
                    html_parts.append(f'<code class="smiles-code">{parsed["smiles"]}</code>')
                    html_parts.append(f'<p style="margin-top: 10px;"><strong>Reasoning:</strong> {parsed["reason"]}</p>')
                    html_parts.append('</div>')
            except (json.JSONDecodeError, TypeError):
                # Not JSON, display as-is
                html_parts.append(f'<pre>{content}</pre>')
            
            html_parts.append('</div>')
        
        else:
            # Unknown role
            html_parts.append('<div class="conv-message">')
            html_parts.append(f'<h3>❓ Message {i+1}: {role}</h3>')
            html_parts.append(f'<pre>{content}</pre>')
            html_parts.append('</div>')
        
        html_parts.append('<hr style="border: 1px solid #e0e0e0; margin: 20px 0;">')
    
    # Summary if available
    if data.get('summary'):
        html_parts.append('<div class="conv-message" style="background-color: #e8f4f8; padding: 20px; border-radius: 8px; border: 2px solid #2196F3;">')
        html_parts.append('<h2 style="color: #1976D2 !important; margin-top: 0;">📊 Final AI-Generated Summary</h2>')
        html_parts.append(f'<div style="color: #000000; line-height: 1.6; white-space: pre-wrap;">{data["summary"]}</div>')
        html_parts.append('</div>')
    
    html_parts.append('</div>')
    
    return ''.join(html_parts)


# Display the formatted conversation
display(HTML(format_conversation_html(data)))